# Lesson 03 - Pattern di Design Agentico

In questa lezione, esploriamo tre pattern di design fondamentali per costruire agenti AI efficaci:

1. **Istruzioni Chiare per l'Agente** — Creare prompt precisi e che definiscano il ruolo per guidare il comportamento dell'agente
2. **Output Strutturato con Modelli Pydantic** — Garantire che gli agenti restituiscano dati prevedibili e convalidati
3. **Agenti con Responsabilità Singola** — Progettare agenti focalizzati che fanno bene ciascuno una singola cosa

Applicheremo ogni pattern a uno scenario di **consulente per destinazioni di viaggio**, costruendo progressivamente un sistema in grado di suggerire destinazioni, verificare disponibilità e gestire la logistica.


## Configurazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pattern 1: Istruzioni chiare per l'agente

Il pattern più efficace è anche il più semplice: scrivere istruzioni chiare e dettagliate per il tuo agente.

Buone istruzioni definiscono:
- **Chi** è l'agente (persona e tono)
- **Cosa** dovrebbe fare (responsabilità passo dopo passo)
- **Come** dovrebbe comportarsi (vincoli e stile)

Qui sotto, creiamo un agente concierge di viaggio con istruzioni esplicite che modellano ogni risposta che produce.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Output strutturato con modelli Pydantic

Il testo libero è utile per la conversazione, ma i sistemi a valle necessitano di dati strutturati.
Abbinando **modelli Pydantic** a una **funzione strumento**, possiamo:

- Definire uno schema esatto per l'output dell'agente
- Validare automaticamente le risposte
- Integrare in modo affidabile i risultati dell'agente nella logica dell'applicazione

Introduciamo anche uno strumento che restituisce i dettagli della destinazione così che l'agente basi le sue raccomandazioni su dati reali.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Agenti a Responsabilità Singola

I compiti complessi traggono vantaggio dalla suddivisione del lavoro tra più agenti focalizzati, ognuno con una singola responsabilità:

- Un **Esperto di Destinazioni** che conosce luoghi e disponibilità
- Un **Pianificatore Logistico** che gestisce voli, hotel e itinerari

Questo rispecchia il principio di ingegneria del software della *separazione delle responsabilità* — ogni agente è più facile da testare, mantenere e migliorare in modo indipendente.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Riepilogo

In questa lezione abbiamo applicato tre modelli di progettazione agentica a uno scenario di raccomandazione di viaggi:

| Pattern | Idea Chiave | Beneficio |
|---|---|---|
| **Istruzioni Chiare** | Definire personaggio, responsabilità e vincoli all'inizio | Comportamento coerente e in linea con il brand dell'agente |
| **Output Strutturato** | Utilizzare modelli Pydantic come formato di risposta | Risultati validati e leggibili dalle macchine |
| **Singola Responsabilità** | Assegnare a ogni agente un compito specifico | Più facile da testare, mantenere e comporre |

Questi modelli si combinano naturalmente — puoi combinare istruzioni chiare con output strutturato all'interno di un agente con singola responsabilità per costruire sistemi robusti e pronti per la produzione.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
